# 02 · Foundation-model baseline

**What this notebook decides.** The reference point that every later claim is
measured against. If this number is wrong, every improvement reported later is
wrong by the same amount.

A baseline is easy to get accidentally low. An unadapted instruction-tuned model
asked for SQL will often return SQL wrapped in prose, or a short explanation
followed by a query. Scored naively, that reads as 0% execution accuracy and
makes any fine-tuned model look transformative. The check for that is the
**valid SQL rate**, not the accuracy — so this notebook looks at it first.

In [ ]:
import pandas as pd

from qwen_text2sql.reporting import (
    artifact,
    environment_report,
    headline_metrics,
    load_records,
    preflight,
    project_root,
)
from qwen_text2sql.reporting.style import use_house_style

ROOT = project_root()
PALETTE = use_house_style("light")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
ROOT

In [ ]:
report = environment_report()
print(f"commit   {report['git_commit']}")
print(f"python   {report['python']}")
print(f"platform {report['platform']}")
pd.Series(report["packages"], name="version").to_frame()

## 1 · What the model actually sees

Prompt construction is where silent evaluation bugs live. This is the exact
string the model is given — schema, optional evidence, question — rendered from
the same code path used in training and inference.

In [ ]:
from qwen_text2sql.io import read_jsonl
from qwen_text2sql.pipeline import prepared_from_mapping
from qwen_text2sql.training.formatting import conversation_prompt

rows = list(read_jsonl(artifact("bird_validation").require()))
example = prepared_from_mapping(rows[0])
rule = "=" * 78
LIMIT = 1500

for message in conversation_prompt(example):
    print(rule)
    print(message["role"].upper())
    print(rule)
    content = message["content"]
    print(content if len(content) <= LIMIT else content[:LIMIT] + " ... truncated for display")
    print()

print(f"Reference SQL: {example.gold_sql}")

## 2 · Generating the baseline

Generation needs an accelerator and downloads a multi-gigabyte checkpoint, so it
is run from the command line rather than from a notebook cell. Notebook state and
a half-hour GPU job are a bad combination: a disconnected kernel loses the run,
and nothing records which cell produced which file.

```bash
poetry run python scripts/generate_predictions.py \
  --config configs/baseline.yaml \
  --data data/processed/bird_validation.jsonl \
  --output results/predictions/baseline_validation.jsonl

poetry run python scripts/evaluate_predictions.py \
  --data data/processed/bird_validation.jsonl \
  --predictions results/predictions/baseline_validation.jsonl \
  --records-output results/baseline_validation_records.jsonl \
  --metrics-output results/baseline_validation_metrics.json
```

Decoding is greedy for the principal comparison, so the baseline is reproducible
rather than a draw from a distribution.

In [ ]:
import pandas as pd

pd.DataFrame(preflight("bird_validation", "baseline_records"))

## 3 · The baseline, with uncertainty

In [ ]:
records = load_records(artifact("baseline_records").require())
headline = headline_metrics(records)
headline.style.format({"estimate": "{:.1%}", "lower": "{:.1%}", "upper": "{:.1%}"}, na_rep="—")

**Read the valid SQL rate before the accuracy.** The two together separate three
very different situations:

| Valid SQL | Execution accuracy | What it means |
|---|---|---|
| high | low | The model writes runnable SQL that answers the wrong question — a genuine reasoning gap |
| low | low | The model is not producing usable SQL at all — likely a prompt or extraction problem, not a capability one |
| low | high on what runs | Extraction is dropping valid queries; fix `sql/extract.py` before trusting anything |

Only the first row is a baseline worth comparing against.

In [ ]:
from qwen_text2sql.reporting import error_breakdown, figures

breakdown = error_breakdown(records)
display(breakdown)
if bool((breakdown["error_kind"] != "none").any()):
    display(figures.error_breakdown(breakdown))

## Reading this baseline

**What it shows.** The unadapted model's execution accuracy on the
database-disjoint validation split, with a 95% Wilson interval, and how its
failures distribute across categories.

**What it does not show.** Nothing here is evidence about fine-tuning. It is
also specific to this prompt: a different schema serialisation or a different
system message would move this number, so the baseline is only comparable to
adapted models that use the identical prompt — which they do, because both call
`qwen_text2sql.training.formatting`.

---

### Provenance

Every number above is produced by code in `src/qwen_text2sql/`, which is unit
tested and type checked. This notebook orchestrates and interprets; it does not
define the analysis. If a figure here looks wrong, the fix belongs in `src/` and
comes with a test.